In [47]:
import pandas as pd
import numpy as np
from pathlib import Path

In [54]:
current = Path.cwd()
project_root = current if (current / "CLAUDE.md").exists() else current.parent.parent

precipitation_review = project_root / "data" / "raw_final" / "precipitation_review.csv"
crop_yield_review = project_root / "data" / "raw_final" / "crop_yield_review.csv"
vegetation_review = project_root / "data" / "raw_final" / "vegetation_review.csv"
geography_common = project_root / "data" / "raw_final" / "geography_common.csv"

In [55]:
current = Path.cwd()
project_root = current if (current / "CLAUDE.md").exists() else current.parent.parent

pickle_folder_path = project_root / "data" / "raw" / "pickle"

crop_yield_df = pd.read_pickle(pickle_folder_path / "df_crop_yield_2016_2026.pkl")
geography_df = pd.read_pickle(pickle_folder_path / "df_geography.pkl")
precipitation_df = pd.read_pickle(pickle_folder_path / "df_precipitation_2016_2026.pkl")
vegetation_df = pd.read_pickle(pickle_folder_path / "df_vegetation_2016_2026.pkl")

In [56]:
crop_yield_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 455359 entries, 0 to 455358
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id               455359 non-null  int64  
 1   year             455359 non-null  object 
 2   state_name       455359 non-null  object 
 3   state_code       455359 non-null  int64  
 4   district_name    455359 non-null  object 
 5   district_code    455359 non-null  int64  
 6   crop_name        455359 non-null  object 
 7   crop_code        455338 non-null  float64
 8   crop_type        455359 non-null  object 
 9   season           455359 non-null  object 
 10  area             455359 non-null  float64
 11  area_unit        455359 non-null  object 
 12  production       450350 non-null  float64
 13  production_unit  455359 non-null  object 
 14  yield            455359 non-null  float64
 15  yield_unit       455359 non-null  object 
dtypes: float64(4), int64(3), object(9)
mem

In [57]:
# renaming the columns

crop_yield_df.rename(
    columns={"state_name": "state", "district_name": "district"}, inplace=True
)
geography_df.rename(columns={"State": "state", "District": "district"}, inplace=True)
precipitation_df.rename(
    columns={"State_Name": "state", "District_Name": "district"}, inplace=True
)
vegetation_df.rename(
    columns={"state_name": "state", "district_name": "district"}, inplace=True
)

In [58]:
crop_yield_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 455359 entries, 0 to 455358
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id               455359 non-null  int64  
 1   year             455359 non-null  object 
 2   state            455359 non-null  object 
 3   state_code       455359 non-null  int64  
 4   district         455359 non-null  object 
 5   district_code    455359 non-null  int64  
 6   crop_name        455359 non-null  object 
 7   crop_code        455338 non-null  float64
 8   crop_type        455359 non-null  object 
 9   season           455359 non-null  object 
 10  area             455359 non-null  float64
 11  area_unit        455359 non-null  object 
 12  production       450350 non-null  float64
 13  production_unit  455359 non-null  object 
 14  yield            455359 non-null  float64
 15  yield_unit       455359 non-null  object 
dtypes: float64(4), int64(3), object(9)
mem

In [59]:
# Determining the common state names and removing unwanted records

states_cy = set(crop_yield_df["state"].unique())
states_geo = set(geography_df["state"].unique())
states_per = set(precipitation_df["state"].unique())
states_veg = set(vegetation_df["state"].unique())

common_states = states_cy & states_geo & states_per & states_veg

common_states_df = pd.DataFrame(list(common_states), columns=["state"])
crop_yield_df = crop_yield_df[crop_yield_df["state"].isin(common_states_df["state"])]
geography_df = geography_df[geography_df["state"].isin(common_states_df["state"])]
precipitation_df = precipitation_df[
    precipitation_df["state"].isin(common_states_df["state"])
]
vegetation_df = vegetation_df[vegetation_df["state"].isin(common_states_df["state"])]

In [60]:
# Cleaning up the districts data. Take geography as the base for the canonical dataset

# crop_yield_df['district'].nunique() 608
# geography_df['district'].nunique() 511
# precipitation_df['district'].nunique() 587
# vegetation_df['district'].nunique() 511

In [61]:
from rapidfuzz import process

common_districts_df = geography_df[["state", "district"]].drop_duplicates()

state_districts = common_districts_df.groupby("state")["district"].apply(list).to_dict()


def district_match(row):
    state = row["state"]
    district = row["district"]

    districts_by_state = state_districts.get(state, [])
    match, score, _ = process.extractOne(district, districts_by_state)
    return pd.Series([match, score])


crop_yield_df[["mapped_district", "score"]] = crop_yield_df.apply(
    district_match, axis=1
)
precipitation_df[["mapped_district", "score"]] = precipitation_df.apply(
    district_match, axis=1
)
vegetation_df[["mapped_district", "score"]] = vegetation_df.apply(
    district_match, axis=1
)

In [71]:
common_districts_df.to_csv(geography_common, index=False)
crop_yield_review_df = crop_yield_df[
    ["state", "district", "mapped_district"]
].drop_duplicates()
crop_yield_review_df.to_csv(crop_yield_review, index=False)
precipitation_review_df = precipitation_df[
    ["state", "district", "mapped_district"]
].drop_duplicates()
precipitation_review_df.to_csv(precipitation_review, index=False)
vegetation_review_df = vegetation_df[
    ["state", "district", "mapped_district"]
].drop_duplicates()
vegetation_review_df.to_csv(vegetation_review, index=False)

KeyError: "['mapped_district'] not in index"

In [69]:
#  manually reviewed the files and updated *_FINAL.csv files with corrected state district combination

precipitation_district_lookup = (
    project_root / "data" / "raw_final" / "precipitation_review_FINAL.csv"
)
crop_yield_district_lookup = (
    project_root / "data" / "raw_final" / "crop_yield_review_FINAL.csv"
)
vegetation_district_lookup = (
    project_root / "data" / "raw_final" / "vegetation_review_FINAL.csv"
)

precipatition_district_lookup_df = pd.read_csv(precipitation_district_lookup)
crop_yield_district_lookup_df = pd.read_csv(crop_yield_district_lookup)
vegetation_district_lookup_df = pd.read_csv(vegetation_district_lookup)

In [73]:
vegetation_df = vegetation_df.merge(
    vegetation_district_lookup_df[["state", "district", "mapped_district"]],
    on=["state", "district"],
    how="left",
)
vegetation_df["district"] = vegetation_df["mapped_district"]
vegetation_df.drop(columns=["mapped_district"], inplace=True)

precipitation_df = precipitation_df.merge(
    precipatition_district_lookup_df[["state", "district", "mapped_district"]],
    on=["state", "district"],
    how="left",
)
precipitation_df["district"] = precipitation_df["mapped_district"]
precipitation_df.drop(columns=["mapped_district"], inplace=True)

crop_yield_df = crop_yield_df.merge(
    crop_yield_district_lookup_df[["state", "district", "mapped_district"]],
    on=["state", "district"],
    how="left",
)
crop_yield_df["district"] = crop_yield_df["mapped_district"]
crop_yield_df.drop(columns=["mapped_district"], inplace=True)

In [75]:
precipitation_processed1 = project_root / "data" / "processed" / "precipitation1.csv"
precipitation_processed2 = project_root / "data" / "processed" / "precipitation2.csv"
precipitation_processed3 = project_root / "data" / "processed" / "precipitation3.csv"
crop_yield_processed = project_root / "data" / "processed" / "crop_yield.csv"
vegetation_processed = project_root / "data" / "processed" / "vegetation.csv"
geography_processed = project_root / "data" / "processed" / "geography.csv"

vegetation_df.to_csv(vegetation_processed)
precipitation_df[:700000].to_csv(precipitation_processed1)
precipitation_df[700000:1400000].to_csv(precipitation_processed2)
precipitation_df[1400000:].to_csv(precipitation_processed3)
crop_yield_df.to_csv(crop_yield_processed)
geography_df.to_csv(geography_processed)


precipitation_processed_pkl1 = (
    project_root / "data" / "processed" / "precipitation1.pkl"
)
precipitation_processed_pkl2 = (
    project_root / "data" / "processed" / "precipitation2.pkl"
)
precipitation_processed_pkl3 = (
    project_root / "data" / "processed" / "precipitation3.pkl"
)
crop_yield_processed_pkl = project_root / "data" / "processed" / "crop_yield.pkl"
vegetation_processed_pkl = project_root / "data" / "processed" / "vegetation.pkl"
geography_processed_pkl = project_root / "data" / "processed" / "geography.pkl"

vegetation_df.to_pickle(vegetation_processed_pkl)
precipitation_df[:700000].to_pickle(precipitation_processed_pkl1)
precipitation_df[700000:1400000].to_pickle(precipitation_processed_pkl2)
precipitation_df[1400000:].to_pickle(precipitation_processed_pkl3)
crop_yield_df.to_pickle(crop_yield_processed_pkl)
geography_df.to_pickle(geography_processed_pkl)